# 10 - Joint (Greedy) Regime-Aware Head Selection and Confidence-Margin Robustness Check

**Project:** Regime-Conditional Attention Head Selection for Time Series Transformers (ReCAHS)

Experiments 05-07 built Policy B (dynamic regime-conditional head selection) by scoring each head **independently** per regime (leave-one-head-out on validation loss) and keeping the top-k heads by that independent score. This notebook addresses two open weaknesses identified in the project so far:

1. **Independent scoring misses head interactions.** Removing head A and removing head B independently may each look harmless, but removing both together can hurt more (or less) than the sum of their individual effects — the top-k-by-independent-score approach cannot see this. This notebook instead uses **greedy backward elimination**: starting from all 24 heads active, it repeatedly removes the single head whose removal least harms *masked* validation MSE, re-evaluating the actual candidate mask at every step (per regime). This is more expensive but captures interactions that independent scoring cannot.
2. **Regime labels are noisier on some windows than others.** STL-based regime labeling produces a `confidence_margin` (gap between the top and second component score); Experiment 04's regime detection notebook already computes this for validation, and Experiment 07 computed it for a one-off test evaluation, but the test-side regime/confidence file was never committed to this repository. This notebook regenerates it (using the identical method as `07_test_regime_detection_dynamic_75_eval.ipynb`) and uses it to check whether dynamic regime-aware selection performs better specifically on **confidently**-labeled windows than on ambiguous ones — a direct test of whether labeling noise is part of why the test-set generalization gap has been so persistent.

Both the existing (independent-score) dynamic 75% keep masks and the new joint (greedy) masks are evaluated side by side, in the same session, against the same baseline and static-pruning references, so all numbers in the final comparison table are directly comparable.

**Expected runtime:** the greedy search evaluates on a random subsample (capped at 256 windows) of each regime's validation windows rather than the full set, to keep runtime manageable; on a T4 GPU the full notebook (STL test-label generation + greedy search + full-set evaluations) takes roughly 45-75 minutes.


## 1. Mount Google Drive and import core libraries

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
import sys
import os
import shutil
import random
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from tqdm.auto import tqdm

## 2. Define project paths

In [4]:
PROJECT_DIR = Path(
    "/content/drive/MyDrive/BIL401_Regime_Head_Pruning"
)

REGIME_DIR = PROJECT_DIR / "regime_detection"
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
HEAD_IMPORTANCE_DIR = PROJECT_DIR / "head_importance"
PRUNING_DIR = PROJECT_DIR / "pruning_experiments"
JOINT_DIR = PRUNING_DIR / "b4_joint_dynamic_75_keep"

for d in [PRUNING_DIR, JOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("REGIME_DIR:", REGIME_DIR)
print("HEAD_IMPORTANCE_DIR:", HEAD_IMPORTANCE_DIR)
print("JOINT_DIR:", JOINT_DIR)

PROJECT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning
REGIME_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/regime_detection
HEAD_IMPORTANCE_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/head_importance
JOINT_DIR: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_joint_dynamic_75_keep


## 3. Set up Time-Series-Library

In [5]:
TSLIB_DIR = Path("/content/Time-Series-Library")

if not TSLIB_DIR.exists():
    %cd /content
    !git clone https://github.com/thuml/Time-Series-Library.git
else:
    print("Time-Series-Library already exists:", TSLIB_DIR)

sys.path.insert(0, str(TSLIB_DIR))

/content
Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 2295, done.
remote: Total 2295 (delta 0), reused 0 (delta 0), pack-reused 2295 (from 1)
Receiving objects: 100% (2295/2295), 78.43 MiB | 24.02 MiB/s, done.
Resolving deltas: 100% (1570/1570), done.


In [6]:
%cd /content/Time-Series-Library
!pip install -q patool sktime scikit-base statsmodels --no-deps
!pip install -q --no-deps einops
!pip install reformer-pytorch --no-deps
!pip install local-attention --no-deps
!pip install hyper_connections --no-deps
!pip install axial_positional_embedding --no-deps
!pip install product_key_memory --no-deps
!pip install colt5_attention --no-deps

/content/Time-Series-Library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 17.3 MB/s eta 0:00:00


## 4. Download the ETTh1 dataset

In [7]:
drive_data_path = PROJECT_DIR / "data" / "ETTh1.csv"

tslib_data_path = (
    TSLIB_DIR
    / "dataset/ETDataset/ETT-small/ETTh1.csv"
)

tslib_data_path.parent.mkdir(parents=True, exist_ok=True)

if drive_data_path.exists():
    shutil.copy2(drive_data_path, tslib_data_path)
    print("ETTh1 copied from Drive.")
else:
    print("Drive data not found. Downloading ETTh1...")
    !wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv -O /content/Time-Series-Library/dataset/ETDataset/ETT-small/ETTh1.csv

df = pd.read_csv(tslib_data_path)
print("Dataset exists:", tslib_data_path.exists(), "| shape:", df.shape)

Drive data not found. Downloading ETTh1...
Dataset exists: True | shape: (17420, 8)


## 5. Load the validation regime labels

In [8]:
regime_path = REGIME_DIR / "etth1_validation_regimes_ot_seq336.csv"
regime_df = pd.read_csv(regime_path)

if "confidence_margin" not in regime_df.columns:
    scores = regime_df[["trend_score", "seasonal_score", "residual_score"]].to_numpy()
    sorted_scores = np.sort(scores, axis=1)[:, ::-1]
    regime_df["confidence_margin"] = sorted_scores[:, 0] - sorted_scores[:, 1]

if "is_confident" not in regime_df.columns:
    regime_df["is_confident"] = regime_df["confidence_margin"] >= 0.05

print("Validation regime df shape:", regime_df.shape)
display(regime_df["regime"].value_counts())
print("Confident windows:", int(regime_df["is_confident"].sum()), "/", len(regime_df))

Validation regime df shape: (2785, 13)


,count
regime,
trend,2134
residual,359
seasonal,292


Confident windows: 2490 / 2785


## 6. Load or regenerate the test regime labels

`etth1_test_regimes_ot_seq336.csv` was produced in Experiment 07 but was never committed to the repository (only its summary numbers made it into `README.md`). If it is not already present on Drive, this cell regenerates it using the exact same method (STL decomposition, `period=24`, same train/val/test split boundaries as `Time-Series-Library`'s `Dataset_ETT_hour`) so the result is consistent with what Experiment 07 originally reported (~93.6% trend / ~4.5% seasonal / ~1.9% residual).

In [9]:
from statsmodels.tsa.seasonal import STL

test_regime_path = REGIME_DIR / "etth1_test_regimes_ot_seq336.csv"

if test_regime_path.exists():
    test_regime_df = pd.read_csv(test_regime_path)
    print("Loaded existing test regime labels:", test_regime_path)
else:
    print("Test regime labels not found on Drive, regenerating...")

    SEQ_LEN = 336
    PRED_LEN = 96
    STL_PERIOD = 24
    TARGET_COL = "OT"

    TRAIN_SIZE = 12 * 30 * 24
    VAL_SIZE = 4 * 30 * 24
    TEST_SIZE = 4 * 30 * 24

    test_border1 = TRAIN_SIZE + VAL_SIZE - SEQ_LEN
    test_border2 = TRAIN_SIZE + VAL_SIZE + TEST_SIZE

    test_segment = df.iloc[test_border1:test_border2].reset_index(drop=True)
    num_test_windows = len(test_segment) - SEQ_LEN - PRED_LEN + 1
    target_values = test_segment[TARGET_COL].to_numpy(dtype=np.float64)

    def safe_variance(values):
        values = np.asarray(values, dtype=np.float64)
        return float(np.var(values)) if len(values) else 0.0

    def label_window_with_stl(series_window, period=24):
        result = STL(series_window, period=period, robust=True).fit()

        trend_var = safe_variance(result.trend)
        seasonal_var = safe_variance(result.seasonal)
        residual_var = safe_variance(result.resid)
        total_var = trend_var + seasonal_var + residual_var

        if total_var <= 1e-12:
            trend_score = seasonal_score = residual_score = 0.0
        else:
            trend_score = trend_var / total_var
            seasonal_score = seasonal_var / total_var
            residual_score = residual_var / total_var

        scores = {"trend": trend_score, "seasonal": seasonal_score, "residual": residual_score}
        regime = max(scores, key=scores.get)
        sorted_scores = sorted(scores.values(), reverse=True)
        confidence_margin = sorted_scores[0] - sorted_scores[1]

        return {
            "regime": regime,
            "trend_score": trend_score,
            "seasonal_score": seasonal_score,
            "residual_score": residual_score,
            "confidence_margin": confidence_margin,
        }

    test_regime_records = []
    for window_id in tqdm(range(num_test_windows), desc="STL test regime labeling"):
        window = target_values[window_id:window_id + SEQ_LEN]
        label_info = label_window_with_stl(window, period=STL_PERIOD)
        test_regime_records.append({"window_id": window_id, **label_info})

    test_regime_df = pd.DataFrame(test_regime_records)
    test_regime_df["is_confident"] = test_regime_df["confidence_margin"] >= 0.05

    test_regime_df.to_csv(test_regime_path, index=False)
    print("Saved regenerated test regime labels to:", test_regime_path)

print("\nTest regime df shape:", test_regime_df.shape)
display(test_regime_df["regime"].value_counts(normalize=True) * 100)
print("Confident windows:", int(test_regime_df["is_confident"].sum()), "/", len(test_regime_df))

Test regime labels not found on Drive, regenerating...


STL test regime labeling:   0%|          | 0/2785 [00:00<?, ?it/s]

Saved regenerated test regime labels to: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/regime_detection/etth1_test_regimes_ot_seq336.csv

Test regime df shape: (2785, 7)


,proportion
regime,
trend,93.572711
seasonal,4.488330
residual,1.938959


Confident windows: 2600 / 2785


## 7. Locate the B4 checkpoint

In [10]:
b4_checkpoint_candidates = list(
    CHECKPOINT_DIR.glob("B4_patchtst_etth1_336_dm128_h8/**/checkpoint.pth")
)

if len(b4_checkpoint_candidates) == 0:
    raise FileNotFoundError(
        "B4 checkpoint bulunamadı. Run notebook 00 first if needed."
    )

b4_checkpoint_path = b4_checkpoint_candidates[0]
print("Selected checkpoint:", b4_checkpoint_path)

Selected checkpoint: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/checkpoints/B4_patchtst_etth1_336_dm128_h8/checkpoint.pth


## 8. Reconstruct the B4 model arguments

In [11]:
from argparse import Namespace

args = Namespace(
    task_name="long_term_forecast",
    is_training=0,
    model_id="ETTh1_336_96_dm128_h8",
    model="PatchTST",

    data="ETTh1",
    root_path="./dataset/ETDataset/ETT-small/",
    data_path="ETTh1.csv",
    features="M",
    target="OT",
    freq="h",
    checkpoints="./checkpoints/",

    seq_len=336,
    label_len=48,
    pred_len=96,
    seasonal_patterns="Monthly",
    inverse=False,

    enc_in=7,
    dec_in=7,
    c_out=7,
    d_model=128,
    n_heads=8,
    e_layers=3,
    d_layers=1,
    d_ff=256,
    moving_avg=25,
    factor=3,
    distil=True,
    dropout=0.1,
    embed="timeF",
    activation="gelu",
    output_attention=False,

    patch_len=16,
    stride=8,
    padding_patch="end",
    revin=1,
    affine=0,
    subtract_last=0,
    decomposition=0,
    kernel_size=25,
    individual=0,

    num_workers=0,
    itr=1,
    train_epochs=10,
    batch_size=32,
    patience=3,
    learning_rate=0.0001,
    des="baseline_b4",
    loss="MSE",
    lradj="type1",
    use_amp=False,
    augmentation_ratio=0.0,

    use_gpu=torch.cuda.is_available(),
    gpu=0,
    use_multi_gpu=False,
    devices="0",
    gpu_type="cuda",
    expand=2,
    d_conv=4,
    top_k=5,
    num_kernels=6,
    channel_independence=0,
    decomp_method="moving_avg",
    use_norm=1,
    down_sampling_layers=0,
    down_sampling_window=1,
    down_sampling_method=None,
    seg_len=48,

    p_hidden_dims=[128, 128],
    p_hidden_layers=2,
)

print(args)

Namespace(task_name='long_term_forecast', is_training=0, model_id='ETTh1_336_96_dm128_h8', model='PatchTST', data='ETTh1', root_path='./dataset/ETDataset/ETT-small/', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=336, label_len=48, pred_len=96, seasonal_patterns='Monthly', inverse=False, enc_in=7, dec_in=7, c_out=7, d_model=128, n_heads=8, e_layers=3, d_layers=1, d_ff=256, moving_avg=25, factor=3, distil=True, dropout=0.1, embed='timeF', activation='gelu', output_attention=False, patch_len=16, stride=8, padding_patch='end', revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0, num_workers=0, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='baseline_b4', loss='MSE', lradj='type1', use_amp=False, augmentation_ratio=0.0, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0', gpu_type='cuda', expand=2, d_conv=4, top_k=5, num_kernels=6, channel_independence=0, decomp_method='moving_

## 9. Load the model and checkpoint weights

In [12]:
from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

exp = Exp_Long_Term_Forecast(args)
model = exp.model.to(device)

checkpoint = torch.load(b4_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint)
model.eval()

print("B4 checkpoint loaded successfully.")

Device: cuda:0
Use GPU: cuda:0
🚀 Lazy Loading: PatchTST ...
B4 checkpoint loaded successfully.


## 10. Build the validation and test loaders

In [13]:
from data_provider.data_factory import data_provider

vali_data, vali_loader = data_provider(args, flag="val")
test_data, test_loader = data_provider(args, flag="test")

print("Validation dataset length:", len(vali_data), "| batches:", len(vali_loader))
print("Test dataset length:", len(test_data), "| batches:", len(test_loader))

assert len(vali_data) == len(regime_df), (len(vali_data), len(regime_df))
assert len(test_data) == len(test_regime_df), (len(test_data), len(test_regime_df))
print("Windows and regime labels match for both validation and test.")

val 2785
test 2785
Validation dataset length: 2785 | batches: 88
Test dataset length: 2785 | batches: 88
Windows and regime labels match for both validation and test.


## 11. DynamicHeadMaskController

Unlike the plain `HeadMaskController` used in notebooks 04/08/09 (a single mask shared by the whole batch), this controller also accepts a **per-window** mask of shape `[batch, num_layers, num_heads]`, so different windows in the same batch can use different regime-specific head subsets. It also accounts for PatchTST's channel-independent encoder, whose internal batch dimension is `original_batch * num_channels` — the per-window mask is repeated across channels to match.

In [14]:
class DynamicHeadMaskController:
    def __init__(self, model, num_channels=7):
        self.model = model
        self.num_channels = num_channels
        self.original_forwards = {}
        self.current_mask = None

    def install(self):
        for layer_idx, encoder_layer in enumerate(self.model.encoder.attn_layers):
            attention_layer = encoder_layer.attention

            if layer_idx in self.original_forwards:
                continue

            original_forward = attention_layer.forward
            self.original_forwards[layer_idx] = original_forward

            def make_masked_forward(layer_idx, attention_layer):
                def masked_forward(
                    queries,
                    keys,
                    values,
                    attn_mask,
                    tau=None,
                    delta=None,
                ):
                    B, L, _ = queries.shape
                    _, S, _ = keys.shape
                    H = attention_layer.n_heads

                    queries_proj = attention_layer.query_projection(queries)
                    keys_proj = attention_layer.key_projection(keys)
                    values_proj = attention_layer.value_projection(values)

                    queries_proj = queries_proj.view(B, L, H, -1)
                    keys_proj = keys_proj.view(B, S, H, -1)
                    values_proj = values_proj.view(B, S, H, -1)

                    out, attn = attention_layer.inner_attention(
                        queries_proj,
                        keys_proj,
                        values_proj,
                        attn_mask,
                        tau=tau,
                        delta=delta,
                    )

                    if self.current_mask is not None:
                        mask = self.current_mask.to(out.device)

                        if mask.ndim == 2:
                            layer_mask = mask[layer_idx].view(1, 1, H, 1)
                        elif mask.ndim == 3:
                            mask_batch = mask.shape[0]
                            if mask_batch != B:
                                if B % mask_batch != 0:
                                    raise ValueError(
                                        f"Cannot expand mask batch {mask_batch} to encoder batch {B}."
                                    )
                                repeat_factor = B // mask_batch
                                mask = mask.repeat_interleave(repeat_factor, dim=0)
                            layer_mask = mask[:, layer_idx, :].view(B, 1, H, 1)
                        else:
                            raise ValueError(f"Unsupported mask shape: {mask.shape}")

                        out = out * layer_mask

                    out = out.view(B, L, -1)

                    return attention_layer.out_projection(out), attn

                return masked_forward

            attention_layer.forward = make_masked_forward(layer_idx, attention_layer)

    def remove(self):
        for layer_idx, original_forward in self.original_forwards.items():
            self.model.encoder.attn_layers[layer_idx].attention.forward = original_forward
        self.original_forwards = {}
        self.current_mask = None

    def set_mask(self, mask):
        self.current_mask = mask.clone().float()

    def set_all_active(self):
        num_layers = len(self.model.encoder.attn_layers)
        num_heads = self.model.encoder.attn_layers[0].attention.n_heads
        self.current_mask = torch.ones(num_layers, num_heads, dtype=torch.float32)

    def build_mask_from_pairs(self, active_pairs, num_layers, num_heads):
        mask = torch.zeros(num_layers, num_heads, dtype=torch.float32)
        for layer_idx, head_idx in active_pairs:
            mask[layer_idx, head_idx] = 1.0
        return mask

    def build_keep_mask_from_df(self, keep_df, num_layers, num_heads):
        mask = torch.zeros(num_layers, num_heads, dtype=torch.float32)
        for _, row in keep_df.iterrows():
            mask[int(row["layer"]), int(row["head"])] = 1.0
        return mask


In [15]:
mask_controller = DynamicHeadMaskController(model)
mask_controller.install()

num_layers = len(model.encoder.attn_layers)
num_heads = model.encoder.attn_layers[0].attention.n_heads
total_heads = num_layers * num_heads
KEEP_RATIO = 0.75
TARGET_KEEP = round(total_heads * KEEP_RATIO)

baseline_mask = torch.ones(num_layers, num_heads)

print("Total heads:", total_heads, "| target active heads per regime (75% keep):", TARGET_KEEP)

Total heads: 24 | target active heads per regime (75% keep): 18


## 12. Evaluation helpers

Two generic functions, parameterized by a `label_df` with a `regime` column aligned to window order, so the same code evaluates both validation (`regime_df`) and test (`test_regime_df`):

- `compute_with_global_mask`: applies one fixed mask to every window (used for the no-pruning baseline and static pruning).
- `compute_dynamic`: applies a different mask per window depending on its regime label (used for both the independent-score and joint/greedy dynamic masks).

In [16]:
mse_criterion = nn.MSELoss(reduction="none")
mae_criterion = nn.L1Loss(reduction="none")


def build_batch_dynamic_mask(window_ids, label_df, regime_masks):
    batch_masks = [regime_masks[label_df.iloc[int(w)]["regime"]] for w in window_ids]
    return torch.stack(batch_masks, dim=0)


def compute_with_global_mask(model, loader, label_df, mask_controller, mask, device, pred_len=96, desc="eval"):
    model.eval()
    mask_controller.set_mask(mask)

    all_records = []
    global_index = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch
            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            outputs = model(batch_x, batch_x_mark, batch_y, batch_y_mark)
            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            batch_size = batch_x.shape[0]
            for i in range(batch_size):
                window_id = global_index + i
                row = label_df.iloc[window_id]
                all_records.append({
                    "window_id": window_id,
                    "regime": row["regime"],
                    "is_confident": bool(row.get("is_confident", True)),
                    "mse": float(mse_per_sample[i].detach().cpu()),
                    "mae": float(mae_per_sample[i].detach().cpu()),
                })
            global_index += batch_size

    result_df = pd.DataFrame(all_records)
    regime_summary = (
        result_df.groupby("regime").agg(mse=("mse", "mean"), mae=("mae", "mean"), count=("window_id", "count")).reset_index()
    )
    overall = {"overall_mse": float(result_df["mse"].mean()), "overall_mae": float(result_df["mae"].mean())}
    return {"overall": overall, "regime_summary": regime_summary, "window_losses": result_df}


def compute_dynamic(model, loader, label_df, regime_masks, mask_controller, device, pred_len=96, desc="dynamic eval"):
    model.eval()

    all_records = []
    global_index = 0

    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch
            batch_x = batch_x.float().to(device)
            batch_y = batch_y.float().to(device)
            batch_x_mark = batch_x_mark.float().to(device)
            batch_y_mark = batch_y_mark.float().to(device)

            batch_size = batch_x.shape[0]
            window_ids = list(range(global_index, global_index + batch_size))
            batch_mask = build_batch_dynamic_mask(window_ids, label_df, regime_masks)
            mask_controller.set_mask(batch_mask)

            outputs = model(batch_x, batch_x_mark, batch_y, batch_y_mark)
            true = batch_y[:, -pred_len:, :]

            mse_per_sample = mse_criterion(outputs, true).mean(dim=(1, 2))
            mae_per_sample = mae_criterion(outputs, true).mean(dim=(1, 2))

            for i in range(batch_size):
                window_id = global_index + i
                row = label_df.iloc[window_id]
                all_records.append({
                    "window_id": window_id,
                    "regime": row["regime"],
                    "is_confident": bool(row.get("is_confident", True)),
                    "mse": float(mse_per_sample[i].detach().cpu()),
                    "mae": float(mae_per_sample[i].detach().cpu()),
                })
            global_index += batch_size

    result_df = pd.DataFrame(all_records)
    regime_summary = (
        result_df.groupby("regime").agg(mse=("mse", "mean"), mae=("mae", "mean"), count=("window_id", "count")).reset_index()
    )
    overall = {"overall_mse": float(result_df["mse"].mean()), "overall_mae": float(result_df["mae"].mean())}
    return {"overall": overall, "regime_summary": regime_summary, "window_losses": result_df}


## 13. Baseline and static-pruning references (recomputed in this session)

In [17]:
static_prune_path = HEAD_IMPORTANCE_DIR / "summaries" / "static_prune_25_percent_heads.csv"
static_prune_df = pd.read_csv(static_prune_path)

static_mask = baseline_mask.clone()
for _, row in static_prune_df.iterrows():
    static_mask[int(row["layer"]), int(row["head"])] = 0.0

baseline_val = compute_with_global_mask(model, vali_loader, regime_df, mask_controller, baseline_mask, device, args.pred_len, "Baseline validation")
baseline_test = compute_with_global_mask(model, test_loader, test_regime_df, mask_controller, baseline_mask, device, args.pred_len, "Baseline test")

static_val = compute_with_global_mask(model, vali_loader, regime_df, mask_controller, static_mask, device, args.pred_len, "Static 25% validation")
static_test = compute_with_global_mask(model, test_loader, test_regime_df, mask_controller, static_mask, device, args.pred_len, "Static 25% test")

print("Baseline   val:", baseline_val["overall"], "| test:", baseline_test["overall"])
print("Static 25% val:", static_val["overall"], "| test:", static_test["overall"])

Baseline validation:   0%|          | 0/88 [00:00<?, ?it/s]

Baseline test:   0%|          | 0/88 [00:00<?, ?it/s]

Static 25% validation:   0%|          | 0/88 [00:00<?, ?it/s]

Static 25% test:   0%|          | 0/88 [00:00<?, ?it/s]

Baseline   val: {'overall_mse': 0.678066410579095, 'overall_mae': 0.5550830314788613} | test: {'overall_mse': 0.37254557146739276, 'overall_mae': 0.3982142798241422}
Static 25% val: {'overall_mse': 0.6536577488778952, 'overall_mae': 0.5506856770446955} | test: {'overall_mse': 0.37831396693815234, 'overall_mae': 0.40442412037609726}


## 14. Reconstruct Experiment 06's independent-score dynamic masks

Rebuilt from `b4_head_importance_all_windows.csv` (the same per-head, per-regime leave-one-out importance table used in notebooks 05-07), so this notebook's "independent" reference is evaluated with the exact same code path as the new joint masks — eliminating any cross-notebook evaluation drift as a confound.

In [18]:
importance_df = pd.read_csv(HEAD_IMPORTANCE_DIR / "b4_head_importance_all_windows.csv")
assert len(importance_df) == total_heads

def get_top_heads_for_regime(importance_df, regime, keep_ratio=0.75):
    importance_col = f"{regime}_importance"
    num_keep = int(len(importance_df) * keep_ratio)
    return importance_df.sort_values(importance_col, ascending=False).head(num_keep)

regime_masks_independent = {}
for regime in ["trend", "seasonal", "residual"]:
    keep_df = get_top_heads_for_regime(importance_df, regime, KEEP_RATIO)
    regime_masks_independent[regime] = mask_controller.build_keep_mask_from_df(keep_df, num_layers, num_heads)
    print(f"{regime}: {int(regime_masks_independent[regime].sum().item())} active heads")

independent_val = compute_dynamic(model, vali_loader, regime_df, regime_masks_independent, mask_controller, device, args.pred_len, "Independent dynamic validation")
independent_test = compute_dynamic(model, test_loader, test_regime_df, regime_masks_independent, mask_controller, device, args.pred_len, "Independent dynamic test")

print("Independent dynamic val:", independent_val["overall"], "| test:", independent_test["overall"])

trend: 18 active heads
seasonal: 18 active heads
residual: 18 active heads


Independent dynamic validation:   0%|          | 0/88 [00:00<?, ?it/s]

Independent dynamic test:   0%|          | 0/88 [00:00<?, ?it/s]

Independent dynamic val: {'overall_mse': 0.6597460945803987, 'overall_mae': 0.5512131252451474} | test: {'overall_mse': 0.37936756071004235, 'overall_mae': 0.4052870982799753}


## 15. Joint (greedy backward elimination) head selection

For each regime, starting from all 24 heads active, repeatedly evaluate removing each still-active head and keep the removal that results in the **lowest** masked MSE on that regime's validation windows, until only `TARGET_KEEP` (18) heads remain. Because each step's candidates are evaluated with the heads removed in *previous* steps already masked out, this captures head interactions that independent per-head scoring cannot.

To keep runtime manageable, each regime's greedy search uses a random (seed-fixed) subsample of up to 256 of its validation windows; final masks are then evaluated on the **full** validation and test sets in the next section.

In [19]:
GREEDY_SUBSAMPLE_SIZE = 256
GREEDY_SEED = 42

def collect_batches(dataset, indices, device, batch_size=32):
    from torch.utils.data import Subset, DataLoader
    subset = Subset(dataset, indices)
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False)
    batches = []
    for batch in loader:
        batch_x, batch_y, batch_x_mark, batch_y_mark = batch
        batches.append((
            batch_x.float().to(device),
            batch_y.float().to(device),
            batch_x_mark.float().to(device),
            batch_y_mark.float().to(device),
        ))
    return batches


def eval_masked_mse_on_batches(model, mask_controller, mask, batches, pred_len):
    model.eval()
    mask_controller.set_mask(mask)
    total_se = 0.0
    total_count = 0
    with torch.no_grad():
        for batch_x, batch_y, batch_x_mark, batch_y_mark in batches:
            outputs = model(batch_x, batch_x_mark, batch_y, batch_y_mark)
            true = batch_y[:, -pred_len:, :]
            se = mse_criterion(outputs, true).mean(dim=(1, 2)).sum().item()
            total_se += se
            total_count += batch_x.shape[0]
    return total_se / total_count


regime_batch_caches = {}
rng = random.Random(GREEDY_SEED)

for regime in ["trend", "seasonal", "residual"]:
    indices = regime_df.index[regime_df["regime"] == regime].tolist()
    if len(indices) > GREEDY_SUBSAMPLE_SIZE:
        indices = sorted(rng.sample(indices, GREEDY_SUBSAMPLE_SIZE))
    regime_batch_caches[regime] = collect_batches(vali_data, indices, device)
    print(f"{regime}: greedy search will use {len(indices)} validation windows")

trend: greedy search will use 256 validation windows
seasonal: greedy search will use 256 validation windows
residual: greedy search will use 256 validation windows


In [20]:
def greedy_backward_eliminate(model, mask_controller, batches, num_layers, num_heads, target_keep, pred_len, desc=""):
    all_heads = [(l, h) for l in range(num_layers) for h in range(num_heads)]
    active = set(all_heads)
    removal_order = []

    while len(active) > target_keep:
        best_head = None
        best_mse = None

        for candidate in active:
            trial_active = active - {candidate}
            mask = torch.zeros(num_layers, num_heads, dtype=torch.float32)
            for (l, h) in trial_active:
                mask[l, h] = 1.0

            mse = eval_masked_mse_on_batches(model, mask_controller, mask, batches, pred_len)

            if best_mse is None or mse < best_mse:
                best_mse = mse
                best_head = candidate

        active.discard(best_head)
        removal_order.append({
            "step": len(removal_order) + 1,
            "layer": best_head[0],
            "head": best_head[1],
            "resulting_mse": best_mse,
            "remaining_heads": len(active),
        })
        print(f"[{desc}] step {len(removal_order)}: removed layer={best_head[0]} head={best_head[1]} "
              f"-> subsample_mse={best_mse:.6f} ({len(active)} heads remain)")

    final_mask = torch.zeros(num_layers, num_heads, dtype=torch.float32)
    for (l, h) in active:
        final_mask[l, h] = 1.0

    return final_mask, pd.DataFrame(removal_order)


regime_masks_joint = {}
removal_order_dfs = {}

for regime in ["trend", "seasonal", "residual"]:
    final_mask, removal_df = greedy_backward_eliminate(
        model=model,
        mask_controller=mask_controller,
        batches=regime_batch_caches[regime],
        num_layers=num_layers,
        num_heads=num_heads,
        target_keep=TARGET_KEEP,
        pred_len=args.pred_len,
        desc=regime,
    )
    regime_masks_joint[regime] = final_mask
    removal_order_dfs[regime] = removal_df
    print(f"\n{regime} final active heads: {int(final_mask.sum().item())}\n")


[trend] step 1: removed layer=1 head=7 -> subsample_mse=0.651886 (23 heads remain)
[trend] step 2: removed layer=1 head=4 -> subsample_mse=0.642320 (22 heads remain)
[trend] step 3: removed layer=0 head=6 -> subsample_mse=0.635309 (21 heads remain)
[trend] step 4: removed layer=2 head=0 -> subsample_mse=0.629376 (20 heads remain)
[trend] step 5: removed layer=1 head=2 -> subsample_mse=0.626092 (19 heads remain)
[trend] step 6: removed layer=0 head=3 -> subsample_mse=0.623379 (18 heads remain)

trend final active heads: 18

[seasonal] step 1: removed layer=1 head=6 -> subsample_mse=0.782845 (23 heads remain)
[seasonal] step 2: removed layer=2 head=5 -> subsample_mse=0.778498 (22 heads remain)
[seasonal] step 3: removed layer=0 head=5 -> subsample_mse=0.773277 (21 heads remain)
[seasonal] step 4: removed layer=0 head=4 -> subsample_mse=0.768931 (20 heads remain)
[seasonal] step 5: removed layer=0 head=3 -> subsample_mse=0.765438 (19 heads remain)
[seasonal] step 6: removed layer=2 head=3

## 16. Evaluate the joint dynamic masks on the full validation and test sets

In [21]:
joint_val = compute_dynamic(model, vali_loader, regime_df, regime_masks_joint, mask_controller, device, args.pred_len, "Joint dynamic validation")
joint_test = compute_dynamic(model, test_loader, test_regime_df, regime_masks_joint, mask_controller, device, args.pred_len, "Joint dynamic test")

print("Joint dynamic val:", joint_val["overall"], "| test:", joint_test["overall"])
display(joint_val["regime_summary"])
display(joint_test["regime_summary"])

Joint dynamic validation:   0%|          | 0/88 [00:00<?, ?it/s]

Joint dynamic test:   0%|          | 0/88 [00:00<?, ?it/s]

Joint dynamic val: {'overall_mse': 0.6535691904556173, 'overall_mae': 0.5467099856859478} | test: {'overall_mse': 0.3774158703949122, 'overall_mae': 0.4028051380837199}


,regime,mse,mae,count
0,residual,0.702128,0.563779,359
1,seasonal,0.680076,0.564095,292
2,trend,0.641773,0.541460,2134


,regime,mse,mae,count
0,residual,0.371882,0.457730,54
1,seasonal,0.348602,0.400555,125
2,trend,0.378913,0.401775,2606


## 17. Confidence-margin robustness check

Splits validation and test windows into **confident** (`confidence_margin >= 0.05`, the same threshold used in `regime_detection.ipynb`) and **ambiguous** windows, and reports MSE for the baseline, static, independent-dynamic and joint-dynamic settings on each subset. If regime-aware selection is being held back by noisy labels on ambiguous windows, its advantage over the baseline/static methods should be larger on the confident subset than on the ambiguous one.

In [22]:
def confidence_breakdown(window_losses_df, setting_name):
    rows = []
    for is_confident, label in [(True, "confident"), (False, "ambiguous")]:
        subset = window_losses_df[window_losses_df["is_confident"] == is_confident]
        if len(subset) == 0:
            continue
        rows.append({
            "setting": setting_name,
            "subset": label,
            "count": len(subset),
            "mse": subset["mse"].mean(),
            "mae": subset["mae"].mean(),
        })
    return rows

confidence_rows = []
for split_name, results in [
    ("validation", {
        "B4_no_pruning": baseline_val,
        "B4_static_pruning_25": static_val,
        "B4_dynamic_75_independent": independent_val,
        "B4_dynamic_75_joint": joint_val,
    }),
    ("test", {
        "B4_no_pruning": baseline_test,
        "B4_static_pruning_25": static_test,
        "B4_dynamic_75_independent": independent_test,
        "B4_dynamic_75_joint": joint_test,
    }),
]:
    for setting_name, result in results.items():
        for row in confidence_breakdown(result["window_losses"], setting_name):
            row["split"] = split_name
            confidence_rows.append(row)

confidence_breakdown_df = pd.DataFrame(confidence_rows)
display(confidence_breakdown_df.sort_values(["split", "subset", "setting"]))

,setting,subset,count,mse,mae,split
13,B4_dynamic_75_independent,ambiguous,185,0.334490,0.398989,test
15,B4_dynamic_75_joint,ambiguous,185,0.330109,0.397055,test
9,B4_no_pruning,ambiguous,185,0.324464,0.389171,test
11,B4_static_pruning_25,ambiguous,185,0.333496,0.396661,test
12,B4_dynamic_75_independent,confident,2600,0.382561,0.405735,test
14,B4_dynamic_75_joint,confident,2600,0.380782,0.403214,test
8,B4_no_pruning,confident,2600,0.375967,0.398858,test
10,B4_static_pruning_25,confident,2600,0.381503,0.404977,test
5,B4_dynamic_75_independent,ambiguous,295,0.692371,0.561126,validation
7,B4_dynamic_75_joint,ambiguous,295,0.626737,0.538982,validation


## 18. Consolidated comparison

In [23]:
summary_df = pd.DataFrame([
    {"setting": "B4_no_pruning", "val_mse": baseline_val["overall"]["overall_mse"], "test_mse": baseline_test["overall"]["overall_mse"]},
    {"setting": "B4_static_pruning_25", "val_mse": static_val["overall"]["overall_mse"], "test_mse": static_test["overall"]["overall_mse"]},
    {"setting": "B4_dynamic_75_independent", "val_mse": independent_val["overall"]["overall_mse"], "test_mse": independent_test["overall"]["overall_mse"]},
    {"setting": "B4_dynamic_75_joint", "val_mse": joint_val["overall"]["overall_mse"], "test_mse": joint_test["overall"]["overall_mse"]},
])

baseline_test_mse = summary_df.loc[summary_df["setting"] == "B4_no_pruning", "test_mse"].iloc[0]
summary_df["test_mse_change_percent"] = (summary_df["test_mse"] - baseline_test_mse) / baseline_test_mse * 100

display(summary_df)

joint_row = summary_df.loc[summary_df["setting"] == "B4_dynamic_75_joint"].iloc[0]
independent_row = summary_df.loc[summary_df["setting"] == "B4_dynamic_75_independent"].iloc[0]

if joint_row["test_mse"] < independent_row["test_mse"]:
    print(f"Joint (greedy) selection improved test MSE over independent scoring: "
          f"{independent_row['test_mse']:.4f} -> {joint_row['test_mse']:.4f}.")
else:
    print(f"Joint (greedy) selection did NOT improve test MSE over independent scoring: "
          f"{independent_row['test_mse']:.4f} -> {joint_row['test_mse']:.4f}.")

,setting,val_mse,test_mse,test_mse_change_percent
0,B4_no_pruning,0.678066,0.372546,0.000000
1,B4_static_pruning_25,0.653658,0.378314,1.548373
2,B4_dynamic_75_independent,0.659746,0.379368,1.831182
3,B4_dynamic_75_joint,0.653569,0.377416,1.307303


Joint (greedy) selection improved test MSE over independent scoring: 0.3794 -> 0.3774.


## 19. Save results

In [24]:
summary_df.to_csv(JOINT_DIR / "consolidated_comparison.csv", index=False)
confidence_breakdown_df.to_csv(JOINT_DIR / "confidence_margin_breakdown.csv", index=False)

for regime in ["trend", "seasonal", "residual"]:
    removal_order_dfs[regime].to_csv(JOINT_DIR / f"{regime}_greedy_removal_order.csv", index=False)

    mask_df = pd.DataFrame(
        regime_masks_joint[regime].numpy(),
        index=[f"layer_{i}" for i in range(num_layers)],
        columns=[f"head_{j}" for j in range(num_heads)],
    )
    mask_df.to_csv(JOINT_DIR / f"{regime}_joint_dynamic_keep_75_mask.csv")

joint_val["regime_summary"].to_csv(JOINT_DIR / "joint_validation_regime_summary.csv", index=False)
joint_test["regime_summary"].to_csv(JOINT_DIR / "joint_test_regime_summary.csv", index=False)
independent_val["regime_summary"].to_csv(JOINT_DIR / "independent_validation_regime_summary.csv", index=False)
independent_test["regime_summary"].to_csv(JOINT_DIR / "independent_test_regime_summary.csv", index=False)

print("Saved all Experiment 10 outputs to:", JOINT_DIR)
for path in sorted(JOINT_DIR.iterdir()):
    print(" -", path.name)

Saved all Experiment 10 outputs to: /content/drive/MyDrive/BIL401_Regime_Head_Pruning/pruning_experiments/b4_joint_dynamic_75_keep
 - confidence_margin_breakdown.csv
 - consolidated_comparison.csv
 - independent_test_regime_summary.csv
 - independent_validation_regime_summary.csv
 - joint_test_regime_summary.csv
 - joint_validation_regime_summary.csv
 - residual_greedy_removal_order.csv
 - residual_joint_dynamic_keep_75_mask.csv
 - seasonal_greedy_removal_order.csv
 - seasonal_joint_dynamic_keep_75_mask.csv
 - trend_greedy_removal_order.csv
 - trend_joint_dynamic_keep_75_mask.csv
